# Prvi stik s podatki — A: prikaz (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 1, Python ekosistem in spoznavanje s surovimi podatki**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_prvi_stik_s_podatki`** (telekom).

Celico zaženete s **Shift + Enter**.

### Ponovitev
- **DataFrame** = tabela: vrstica je opazovanje, stolpec je spremenljivka. En stolpec je **Series**.
- **Vektorizacija**: operacijo zapišemo za cel stolpec naenkrat — brez `for` zanke.
- Štirje ukazi za vsak nov nabor podatkov: `pd.read_csv()`, `.shape`, `.head()`, `.info()` / `.dtypes`.
- **Tip v Pandasu ni isto kot tip po pomenu.** Pandas ne ve, da je poštna številka oznaka in da je `odhod` skupina.
  Pogoste napake: datum kot besedilo (`object` / `str`), poštna številka kot število, celo število kot `float`.
- Cilj: **ena vrstica = eno opazovanje** (stranka, naročilo, …), en stolpec = ena lastnost.

### Nastavitev okolja
Ta celica poskrbi, da Python najde datoteko `utils/orodja.py` s pomožnimi funkcijami.
V Google Colabu najprej prenese gradivo z GitHuba. V naslednjih zvezkih je ista celica skrita.

In [ ]:
# Nastavitev okolja
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

## A0. Prenos podatkov s Kaggla
Oba nabora podatkov sta javno objavljena na Kagglu. Knjižnica `kagglehub` ju prenese (prijava ni potrebna)
in vrne mapo, kamor ju je shranila. Ob drugem zagonu ju vzame iz predpomnilnika.

In [ ]:
import kagglehub

trgovina_pot = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
telekom_pot = Path(kagglehub.dataset_download("abhinav89/telecom-customer"))

print(trgovina_pot)
sorted(f.name for f in trgovina_pot.glob("*.csv"))

Spletna trgovina ima **9 povezanih tabel**. Telekom ima eno samo tabelo — a koliko stolpcev?

In [ ]:
import pandas as pd

telekom_surovo = pd.read_csv(telekom_pot / "Telecom_customer churn.csv")
print(telekom_surovo.shape)
telekom_surovo.columns[:20].tolist()        # prvih 20 imen stolpcev

**100 stolpcev** s kratkimi angleškimi imeni (`mou_Mean`, `totmrc_Mean`, `eqpdays` …).
Za delavnico smo izbrali **22 stolpcev** in jim dali slovenska imena.
Seznam je v `utils/orodja.py` kot slovar `TELEKOM_STOLPCI`: slovensko ime → (izvirno ime, opis).

In [ ]:
from utils.orodja import TELEKOM_STOLPCI

nova_imena = {izvirno: slovensko for slovensko, (izvirno, opis) in TELEKOM_STOLPCI.items()}
telekom = telekom_surovo[list(nova_imena)].rename(columns=nova_imena)   # izberemo in preimenujemo
print(telekom.shape)
telekom.head(3)

Vse to — prenos, izbor stolpcev in shranjevanje v mapo `data/` — smo zapakirali v **funkcijo**
`poskrbi_za_podatke()` v `utils/orodja.py`. V vseh naslednjih zvezkih jo samo pokličemo:

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

---
# Del A — Prikaz: spletna trgovina
Podatki brazilske spletne trgovine Olist: ~100 000 naročil (2016–2018), razdeljenih v več povezanih tabel
(naročila, postavke, plačila, stranke, izdelki, ocene).

## A1. Knjižnice

In [ ]:
import numpy as np
import pandas as pd

## A2. Naložimo tabelo naročil
`pd.read_csv` prebere CSV datoteko v **DataFrame** (tabelo).

In [ ]:
narocila = pd.read_csv(DATA / "trgovina" / "olist_orders_dataset.csv")
narocila.shape          # (število vrstic, število stolpcev)

In [ ]:
narocila.head()         # prvih 5 vrstic

In [ ]:
narocila.info()         # tipi stolpcev in število ne-manjkajočih vrednosti

**Kaj opazimo?**
- Vsi datumi (`order_purchase_timestamp`, …) so tipa `object` / `str` — torej **besedilo**, ne datum.
  Z besedilom ne moremo računati (npr. koliko dni je trajala dostava).
- Nekateri datumi imajo manj ne-manjkajočih vrednosti kot ima tabela vrstic → **manjkajoče vrednosti** (popoldne).

## A3. Popravimo tip: besedilo → datum

In [ ]:
datumi = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]
narocila = pd.read_csv(DATA / "trgovina" / "olist_orders_dataset.csv", parse_dates=datumi)
narocila.dtypes

Zdaj lahko z datumi računamo — na primer, koliko dni je trajala dostava.
Ta stolpec bomo kasneje tudi **napovedovali** (regresija).

In [ ]:
narocila["dni_dostave"] = (narocila["order_delivered_customer_date"]
                           - narocila["order_purchase_timestamp"]).dt.days
narocila["dni_dostave"].describe()

## A4. Številka, ki ni številka: poštna številka

In [ ]:
stranke = pd.read_csv(DATA / "trgovina" / "olist_customers_dataset.csv")
stranke.dtypes

In [ ]:
stranke.head(3)

Kaj v resnici piše v datoteki? Preberimo prve 3 vrstice kot navadno besedilo:

In [ ]:
with open(DATA / "trgovina" / "olist_customers_dataset.csv", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip())

V datoteki piše `"09790"`, Pandas pa je prebral število `9790` — **vodilna ničla je izginila**.
Poštna številka je **oznaka**, ne količina: z njo ne računamo povprečja.
Pandasu povemo, naj stolpec prebere kot besedilo:

In [ ]:
stranke = pd.read_csv(DATA / "trgovina" / "olist_customers_dataset.csv",
                      dtype={"customer_zip_code_prefix": str})
stranke.head(3)

**Kaj je ena vrstica?** Trgovina ob vsakem naročilu ustvari **nov** `customer_id`.
Isto osebo poveže šele `customer_unique_id`:

In [ ]:
print("Vrstic:                       ", len(stranke))
print("Različnih customer_id:        ", stranke["customer_id"].nunique())
print("Različnih customer_unique_id: ", stranke["customer_unique_id"].nunique())

Ena vrstica = en `customer_id` = eno naročilo. Za zgodovino nakupov ene osebe (npr. kasneje pri segmentaciji strank)
moramo uporabiti `customer_unique_id`. **Pri vsaki tabeli najprej preverimo, kaj pomeni ena vrstica.**

## A5. Zakaj ne `for` zanke? Vektorizacija
Ceni vsake postavke prištejemo 22 % DDV — enkrat z zanko, enkrat z enim ukazom nad celim stolpcem.

In [ ]:
import time
postavke = pd.read_csv(DATA / "trgovina" / "olist_order_items_dataset.csv")
print(f"Število postavk: {len(postavke):,}   različnih naročil: {postavke['order_id'].nunique():,}")

zacetek = time.perf_counter()
z_ddv = []
for cena in postavke["price"]:          # Python zanka: vrstico za vrstico
    z_ddv.append(cena * 1.22)
cas_zanka = time.perf_counter() - zacetek

zacetek = time.perf_counter()
z_ddv_vekt = postavke["price"] * 1.22   # vektorizirano: cel stolpec naenkrat
cas_vekt = time.perf_counter() - zacetek

print(f"Zanka:          {cas_zanka * 1000:8.2f} ms")
print(f"Vektorizirano:  {cas_vekt * 1000:8.2f} ms   (~{cas_zanka / cas_vekt:.0f}x hitreje)")

Mimogrede: postavk je več kot naročil. **Ena vrstica = ena postavka**, eno naročilo ima lahko več postavk.
Ko tabele združujemo, moramo postavke najprej sešteti po naročilu.

## A6. Iz pol-strukturiranih podatkov v tabelo
Sistemi pogosto beležijo dogodke v obliki **JSON** — vsak zapis ima lahko druga polja.
`pd.json_normalize` jih razgrne v tabelo. Kjer polja ni, dobimo `NaN`.

In [ ]:
dogodki = [
    {"id_stranke": 1001, "cas": "2026-11-02 08:15", "tip": "klic",
     "podrobnosti": {"trajanje_s": 125, "smer": "odhodni"}},
    {"id_stranke": 1001, "cas": "2026-11-02 12:40", "tip": "sms",
     "podrobnosti": {"dolzina_znakov": 42}},
    {"id_stranke": 1002, "cas": "2026-11-03 19:05", "tip": "klic",
     "podrobnosti": {"trajanje_s": 610, "smer": "dohodni", "gostovanje": True}},
]
pd.json_normalize(dogodki)

## A7. Iz nestrukturiranih podatkov v značilko
Komentar kupca je prosto besedilo. Najpreprostejša pretvorba v število: **dolžina komentarja**.
(Naprednejše: kategorija pritožbe, sentiment, … )

In [ ]:
ocene = pd.read_csv(DATA / "trgovina" / "olist_order_reviews_dataset.csv")
ocene["dolzina_komentarja"] = ocene["review_comment_message"].str.len()
ocene[["review_score", "review_comment_message", "dolzina_komentarja"]].dropna().head()

---
Naprej: zvezek **`B_prvi_stik_s_podatki`** — isto naredite sami na podatkih telekoma.